In [2]:
import pandas as pd
import numpy as np

orders = pd.read_csv("../data/raw/orders.csv")

orders_clean = orders.copy()

print("Orders Shape:", orders_clean.shape)

print("\nFirst 5 Records:")
print(orders_clean.head())

print("\nDataset Information:")
orders_clean.info()

Orders Shape: (13084, 9)

First 5 Records:
     order_id customer_id product_id  quantity             order_ts  \
0  ORD-108250       C0792       P004         2  2025-09-01 12:24:59   
1  ORD-101939       C0190       P003         1  2025-08-17 07:07:23   
2  ORD-108765       C0841       P033         1  2025-07-13 09:42:32   
3  ORD-108861       C0852       P010         1  2025-07-04 11:23:24   
4  ORD-109150       C0877       P018         2  2025-05-07 15:35:01   

      status  discount_pct order_amount  delivery_minutes  
0  completed             0        310.0              17.0  
1  completed             5       232.75              21.0  
2  completed            10       1349.1              27.0  
3  completed             0         45.0              30.0  
4  completed            20         96.0              28.0  

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 13084 entries, 0 to 13083
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
--- 

Check Missing Values

In [3]:
print("Missing Values in Each Column:")
print(orders_clean.isnull().sum())

print("\nMissing Value Percentage:")
print(
    (orders_clean.isnull().sum() / len(orders_clean)) * 100
)

Missing Values in Each Column:
order_id               0
customer_id            0
product_id             0
quantity               0
order_ts               0
status                 0
discount_pct           0
order_amount          53
delivery_minutes    2541
dtype: int64

Missing Value Percentage:
order_id             0.000000
customer_id          0.000000
product_id           0.000000
quantity             0.000000
order_ts             0.000000
status               0.000000
discount_pct         0.000000
order_amount         0.405075
delivery_minutes    19.420666
dtype: float64


Duplicate Audit

In [4]:
print("Duplicate Rows:",
      orders_clean.duplicated().sum())

print("Duplicate Order IDs:",
      orders_clean["order_id"].duplicated().sum())

Duplicate Rows: 119
Duplicate Order IDs: 775


Investigate Duplicate Order IDs

In [5]:
# Find repeated Order IDs
duplicate_orders = orders_clean[
    orders_clean["order_id"].duplicated(keep=False)
]

print("Total Records with Repeated Order IDs:",
      len(duplicate_orders))

print("\nFirst 10 Duplicate Order Records:")
print(
    duplicate_orders
    .sort_values("order_id")
    .head(10)
)

Total Records with Repeated Order IDs: 1547

First 10 Duplicate Order Records:
         order_id customer_id product_id  quantity             order_ts  \
5467   ORD-100004       C0001       P023         2  2025-07-29 22:27:32   
8195   ORD-100004       C0001       P023         2  2025-07-29 22:17:32   
307    ORD-100010       C0002       P039         2  2025-06-21 17:06:12   
10130  ORD-100010       C0002       P039         2  2025-06-21 17:12:12   
7229   ORD-100020       C0003       P013         2  2025-09-06 13:49:25   
10846  ORD-100020       C0003       P013         2  2025-09-06 13:45:25   
3737   ORD-100029       C0003       P006         2  2025-09-22 09:10:11   
10064  ORD-100029       C0003       P006         2  2025-09-22 09:01:11   
5657   ORD-100050       C0007       P004         2  2025-07-13 22:11:52   
5248   ORD-100050       C0007       P004         2  2025-07-13 22:11:52   

          status  discount_pct order_amount  delivery_minutes  
5467   completed            20 

Count Repeated Order IDs

In [6]:
order_id_counts = (
    orders_clean["order_id"]
    .value_counts()
)

print(
    order_id_counts[order_id_counts > 1].head(10)
)

order_id
ORD-108807    3
ORD-111253    3
ORD-100107    3
ORD-110729    2
ORD-107731    2
ORD-103530    2
ORD-109227    2
ORD-109032    2
ORD-109716    2
ORD-111170    2
Name: count, dtype: int64


Duplicate Order ID investigation

In [7]:
duplicate_summary = (
    duplicate_orders.groupby("order_id")
    .agg(
        record_count=("order_id", "size"),
        customer_count=("customer_id", "nunique"),
        product_count=("product_id", "nunique"),
        status_count=("status", "nunique"),
        timestamp_count=("order_ts", "nunique")
    )
)

print("Duplicate Order ID Groups:",
      len(duplicate_summary))

print("\nDuplicate Summary:")
print(duplicate_summary.head(15))

Duplicate Order ID Groups: 772

Duplicate Summary:
            record_count  customer_count  product_count  status_count  \
order_id                                                                
ORD-100004             2               1              1             2   
ORD-100010             2               1              1             2   
ORD-100020             2               1              1             2   
ORD-100029             2               1              1             2   
ORD-100050             2               1              1             1   
ORD-100062             2               1              1             2   
ORD-100073             2               1              1             2   
ORD-100080             2               1              1             2   
ORD-100084             2               1              1             2   
ORD-100097             2               1              1             2   
ORD-100107             3               1              1             2   


Check Duplicate Groups Across All Records

In [8]:
print("Order IDs with Different Status:")
print((duplicate_summary["status_count"] > 1).sum())

print("\nOrder IDs with Different Customers:")
print((duplicate_summary["customer_count"] > 1).sum())

print("\nOrder IDs with Different Products:")
print((duplicate_summary["product_count"] > 1).sum())

print("\nOrder IDs with Different Timestamps:")
print((duplicate_summary["timestamp_count"] > 1).sum())

Order IDs with Different Status:
656

Order IDs with Different Customers:
0

Order IDs with Different Products:
0

Order IDs with Different Timestamps:
656


Investigate the Remaining 116 Groups

In [9]:
same_status_timestamp = duplicate_summary[
    (duplicate_summary["status_count"] == 1) &
    (duplicate_summary["timestamp_count"] == 1)
]

print("Same Status and Timestamp Groups:",
      len(same_status_timestamp))

print("\nSample Records:")

print(
    duplicate_orders[
        duplicate_orders["order_id"].isin(
            same_status_timestamp.index
        )
    ].sort_values("order_id").head(15)
)

Same Status and Timestamp Groups: 116

Sample Records:
        order_id customer_id product_id  quantity             order_ts  \
5657  ORD-100050       C0007       P004         2  2025-07-13 22:11:52   
5248  ORD-100050       C0007       P004         2  2025-07-13 22:11:52   
1724  ORD-100261       C0033       P016         3  2025-09-03 19:38:33   
802   ORD-100261       C0033       P016         3  2025-09-03 19:38:33   
70    ORD-100474       C0058       P016         3  2025-09-05 15:50:05   
3473  ORD-100474       C0058       P016         3  2025-09-05 15:50:05   
9174  ORD-100649       C0070       P002         1  2025-04-05 18:58:57   
4991  ORD-100649       C0070       P002         1  2025-04-05 18:58:57   
2543  ORD-100761       C0075       P001         3  2025-09-01 10:56:14   
9554  ORD-100761       C0075       P001         3  2025-09-01 10:56:14   
2135  ORD-100784       C0078       P006         1  2025-09-13 12:04:22   
6440  ORD-100784       C0078       P006         1  2025-0

Verify Exact Duplicate Records

In [10]:
same_status_records = duplicate_orders[
    duplicate_orders["order_id"].isin(
        same_status_timestamp.index
    )
]

print("Total Records in 116 Groups:",
      len(same_status_records))

print("\nExact Duplicate Rows in These Groups:",
      same_status_records.duplicated().sum())

print("\nUnique Order IDs:",
      same_status_records["order_id"].nunique())

Total Records in 116 Groups: 232

Exact Duplicate Rows in These Groups: 116

Unique Order IDs: 116


Remove Exact Duplicate Rows

In [11]:
orders_clean = orders_clean.drop_duplicates()

print("Shape After Removing Exact Duplicates:",
      orders_clean.shape)

print("\nRemaining Duplicate Rows:",
      orders_clean.duplicated().sum())

Shape After Removing Exact Duplicates: (12965, 9)

Remaining Duplicate Rows: 0


Data Types Audit

In [12]:
print("Current Data Types:")
print(orders_clean.dtypes)

print("\nOrder Amount Unique Values:")
print(orders_clean["order_amount"].unique()[:20])

print("\nOrder Timestamp Sample:")
print(orders_clean["order_ts"].head())

Current Data Types:
order_id                str
customer_id             str
product_id              str
quantity              int64
order_ts                str
status                  str
discount_pct          int64
order_amount            str
delivery_minutes    float64
dtype: object

Order Amount Unique Values:
<StringArray>
[ '310.0', '232.75', '1349.1',   '45.0',   '96.0',  '106.4',  '300.0',
   '68.4',   '54.0',   '29.0',  '639.2',  '100.8',  '126.0',  '153.0',
  '157.5',  '297.0',   '90.0',  '478.8',  '309.6',  '210.0']
Length: 20, dtype: str

Order Timestamp Sample:
0    2025-09-01 12:24:59
1    2025-08-17 07:07:23
2    2025-07-13 09:42:32
3    2025-07-04 11:23:24
4    2025-05-07 15:35:01
Name: order_ts, dtype: str


Validate Before Conversion

In [13]:
amount_check = pd.to_numeric(
    orders_clean["order_amount"],
    errors="coerce"
)

invalid_amounts = orders_clean[
    orders_clean["order_amount"].notna() &
    amount_check.isna()
]

print("Missing Order Amounts:",
      orders_clean["order_amount"].isnull().sum())

print("Invalid Non-Numeric Amounts:",
      len(invalid_amounts))

print("\nInvalid Amount Samples:")
print(invalid_amounts["order_amount"].head())


# Check invalid timestamps
timestamp_check = pd.to_datetime(
    orders_clean["order_ts"],
    errors="coerce"
)

invalid_timestamps = orders_clean[
    orders_clean["order_ts"].notna() &
    timestamp_check.isna()
]

print("\nInvalid Timestamps:",
      len(invalid_timestamps))

Missing Order Amounts: 53
Invalid Non-Numeric Amounts: 65

Invalid Amount Samples:
187    error
253    error
282    error
498    error
500    error
Name: order_amount, dtype: str

Invalid Timestamps: 0


Identify All Invalid Amount Values

In [14]:
print("Invalid Amount Value Counts:")

print(
    invalid_amounts["order_amount"]
    .value_counts(dropna=False)
)

Invalid Amount Value Counts:
order_amount
error    65
Name: count, dtype: int64


Investigate the 65 Error Records

In [15]:
print("Invalid Amount Records:")

print(
    orders_clean.loc[
        orders_clean["order_amount"].eq("error"),
        [
            "order_id",
            "product_id",
            "quantity",
            "discount_pct",
            "order_amount",
            "status"
        ]
    ].head(10)
)

Invalid Amount Records:
        order_id product_id  quantity  discount_pct order_amount     status
187   ORD-102931       P017         1            25        error  completed
253   ORD-101709       P033         1             0        error  completed
282   ORD-103504       P038         1            40        error  completed
498   ORD-101628       P009         2            40        error  completed
500   ORD-109149       P003         1            40        error    pending
674   ORD-101402       P020         2             0        error  completed
765   ORD-102409       P009         1            20        error  completed
921   ORD-102782       P001         2            25        error  completed
1168  ORD-106222       P019         1            15        error  completed
1186  ORD-105152       P013         2             5        error  completed


Verify Order Amount Calculation 

Expected Amount =  Products Price  *  Quantity (1 - Discount /100  )

so produts_cleaned.csv load 

In [64]:
# Load product reference data

products_ref = pd.read_csv("../data/raw/products.csv")

# Convert valid order amounts to numeric
valid_amounts = pd.to_numeric(
    orders_clean["order_amount"],
    errors="coerce"
)

# Join product prices with orders
check_df = orders_clean.copy()
check_df["actual_amount"] = valid_amounts

check_df = check_df.merge(
    products_ref[["product_id", "price_inr"]],
    on="product_id",
    how="left"
)

# Calculate expected amount
check_df["expected_amount"] = (
    check_df["price_inr"]
    * check_df["quantity"]
    * (1 - check_df["discount_pct"] / 100)
)

# Compare actual and expected amounts
valid_rows = check_df[
    check_df["actual_amount"].notna()
]

matches = (
    valid_rows["actual_amount"]
    - valid_rows["expected_amount"]
).abs() < 0.01



In [65]:
print("Valid Amount Records:", len(valid_rows))
print("Matching Amounts:", matches.sum())
print("Non-Matching Amounts:", (~matches).sum())
print("Missing Product Prices:",
      check_df["price_inr"].isnull().sum())

Valid Amount Records: 12197
Matching Amounts: 12197
Non-Matching Amounts: 0
Missing Product Prices: 0


Handling Missing and Invalid Order Amounts

In [18]:
print("Missing Amounts:", orders_clean["order_amount"].isna().sum())

print(
    "Error Amounts:",
    (orders_clean["order_amount"] == "error").sum()
)

Missing Amounts: 53
Error Amounts: 65


Convert invalid values into NaN

In [19]:
orders_clean["order_amount"] = pd.to_numeric(
    orders_clean["order_amount"],
    errors="coerce"
)

In [20]:
print("Missing Amounts:", orders_clean["order_amount"].isna().sum())

Missing Amounts: 118


Recovering Missing Order Amounts Using Product Prices

In [21]:
missing_amounts = orders_clean[
    orders_clean["order_amount"].isna()
]

print(missing_amounts.shape)
missing_amounts.head(10)

(118, 9)


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
40,ORD-100918,C0086,P012,3,2025-04-25 11:20:28,completed,5,NaN,8.0
187,ORD-102931,C0272,P017,1,2025-08-01 11:13:03,completed,25,NaN,48.0
253,ORD-101709,C0166,P033,1,2025-08-13 22:19:21,completed,0,NaN,67.0
282,ORD-103504,C0331,P038,1,2025-04-05 18:36:19,completed,40,NaN,NaN
354,ORD-104699,C0444,P018,2,2025-08-28 08:49:22,completed,0,NaN,25.0
498,ORD-101628,C0159,P009,2,2025-09-15 08:05:51,completed,40,NaN,25.0
500,ORD-109149,C0877,P003,1,2025-04-12 18:34:33,pending,40,NaN,NaN
674,ORD-101402,C0138,P020,2,2025-09-16 18:57:00,completed,0,NaN,31.0
765,ORD-102409,C0229,P009,1,2025-09-10 14:36:03,completed,20,NaN,23.0
921,ORD-102782,C0258,P001,2,2025-04-22 18:03:50,completed,25,NaN,22.0


Calculating Expected Order Amounts

In [22]:
products_ref = pd.read_csv("../data/raw/products.csv")

Merge Products with missing Orders

In [23]:
missing_check = missing_amounts.merge(
    products_ref[["product_id", "price_inr"]],
    on="product_id",
    how="left"
)

Calculate expected amount

In [24]:
missing_check["expected_amount"] = (
    missing_check["price_inr"] *
    missing_check["quantity"] *
    (1 - missing_check["discount_pct"] / 100)
)

View the results

In [25]:
missing_check[
    ["order_id", "product_id", "quantity",
     "discount_pct", "price_inr", "expected_amount"]
].head(10)

,order_id,product_id,quantity,discount_pct,price_inr,expected_amount
0,ORD-100918,P012,3,5,58,165.3
1,ORD-102931,P017,1,25,20,15.0
2,ORD-101709,P033,1,0,1499,1499.0
3,ORD-103504,P038,1,40,1999,1199.4
4,ORD-104699,P018,2,0,60,120.0
5,ORD-101628,P009,2,40,29,34.8
6,ORD-109149,P003,1,40,245,147.0
7,ORD-101402,P020,2,0,150,300.0
8,ORD-102409,P009,1,20,29,23.2
9,ORD-102782,P001,2,25,129,193.5


Check order status distribution

In [26]:
missing_check["status"].value_counts()

status
completed    112
pending        6
Name: count, dtype: int64

Identify duplicate Order IDs

In [27]:
duplicate_missing = missing_check[
    missing_check["order_id"].duplicated(keep=False)
]

print("Duplicate records:", len(duplicate_missing))

duplicate_missing[
    ["order_id", "order_ts", "status", "expected_amount"]
].sort_values("order_id")

Duplicate records: 12


,order_id,order_ts,status,expected_amount
76,ORD-100240,2025-07-17 09:07:30,completed,31.5
109,ORD-100240,2025-07-17 09:02:30,pending,31.5
41,ORD-100441,2025-08-08 09:05:59,pending,216.0
46,ORD-100441,2025-08-08 09:09:59,completed,216.0
52,ORD-101209,2025-04-02 09:28:14,completed,360.0
57,ORD-101209,2025-04-02 09:20:14,pending,360.0
91,ORD-108448,2025-06-04 07:36:49,completed,84.0
98,ORD-108448,2025-06-04 07:34:49,pending,84.0
29,ORD-109049,2025-08-25 11:26:01,completed,72.0
48,ORD-109049,2025-08-25 11:24:01,pending,72.0


In [28]:
duplicate_missing[
    [
        "order_id",
        "customer_id",
        "product_id",
        "quantity",
        "order_ts",
        "status",
        "discount_pct",
        "expected_amount"
    ]
].sort_values("order_id")

,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,expected_amount
76,ORD-100240,C0031,P008,1,2025-07-17 09:07:30,completed,25,31.5
109,ORD-100240,C0031,P008,1,2025-07-17 09:02:30,pending,25,31.5
41,ORD-100441,C0052,P024,3,2025-08-08 09:05:59,pending,20,216.0
46,ORD-100441,C0052,P024,3,2025-08-08 09:09:59,completed,20,216.0
52,ORD-101209,C0119,P007,3,2025-04-02 09:28:14,completed,0,360.0
57,ORD-101209,C0119,P007,3,2025-04-02 09:20:14,pending,0,360.0
91,ORD-108448,C0805,P015,1,2025-06-04 07:36:49,completed,0,84.0
98,ORD-108448,C0805,P015,1,2025-06-04 07:34:49,pending,0,84.0
29,ORD-109049,C0870,P007,1,2025-08-25 11:26:01,completed,40,72.0
48,ORD-109049,C0870,P007,1,2025-08-25 11:24:01,pending,40,72.0


Recover Completed Order Amounts

In [29]:
completed_missing = missing_check[
    missing_check["status"] == "completed"
]

orders_clean.loc[
    completed_missing.index, "order_amount"
] = completed_missing["expected_amount"]

In [30]:
print("Recovered Amounts:",
      orders_clean.loc[completed_missing.index, "order_amount"].notna().sum())

print("Remaining Missing Amounts:",
      orders_clean["order_amount"].isna().sum())

Recovered Amounts: 112
Remaining Missing Amounts: 117


In [31]:
orders_clean = orders.copy()

orders_clean = orders_clean.drop_duplicates()

orders_clean["order_amount"] = pd.to_numeric(
    orders_clean["order_amount"],
    errors="coerce"
)

In [32]:
missing_amounts = orders_clean[
    orders_clean["order_amount"].isna()
].copy()

missing_amounts["_source_index"] = missing_amounts.index

In [33]:
missing_check = missing_amounts.merge(
    products_ref[["product_id", "price_inr"]],
    on="product_id",
    how="left"
)

missing_check["expected_amount"] = (
    missing_check["price_inr"] *
    missing_check["quantity"] *
    (1 - missing_check["discount_pct"] / 100)
)

In [34]:
completed_missing = missing_check[
    missing_check["status"] == "completed"
]

orders_clean.loc[
    completed_missing["_source_index"],
    "order_amount"
] = completed_missing["expected_amount"].to_numpy()

In [35]:
print(
    "Remaining Missing Amounts:",
    orders_clean["order_amount"].isna().sum()
)

Remaining Missing Amounts: 6


Handling Pending Orders

In [36]:
orders_clean.loc[
    orders_clean["order_amount"].isna(),
    ["order_id", "status", "order_amount"]
]

,order_id,status,order_amount
500,ORD-109149,pending,NaN
4297,ORD-100441,pending,NaN
5228,ORD-109049,pending,NaN
6665,ORD-101209,pending,NaN
10746,ORD-108448,pending,NaN
12183,ORD-100240,pending,NaN


Convert the column

In [37]:
orders_clean["order_ts"] = pd.to_datetime(
    orders_clean["order_ts"],
    errors="coerce"
)

In [38]:
print(orders_clean["order_ts"].dtype)

datetime64[us]


Standardizing Order Status

In [39]:
orders_clean["status"].value_counts()

status
completed    11110
cancelled     1199
pending        656
Name: count, dtype: int64

Validate Quantity and Discount

In [40]:
print("Quantity Summary:")
print(orders_clean["quantity"].describe())

print("\nDiscount Summary:")
print(orders_clean["discount_pct"].describe())

Quantity Summary:
count    12965.000000
mean         1.884690
std          0.783715
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max          3.000000
Name: quantity, dtype: float64

Discount Summary:
count    12965.000000
mean        12.917470
std         13.031723
min          0.000000
25%          0.000000
50%         10.000000
75%         20.000000
max         40.000000
Name: discount_pct, dtype: float64


Verify invalid values

In [41]:
invalid_quantity = orders_clean[
    orders_clean["quantity"] <= 0
]

invalid_discount = orders_clean[
    (orders_clean["discount_pct"] < 0) |
    (orders_clean["discount_pct"] > 100)
]

print("Invalid Quantity:", len(invalid_quantity))
print("Invalid Discount:", len(invalid_discount))

Invalid Quantity: 0
Invalid Discount: 0


Delivery Time Validation

In [42]:
orders_clean.groupby("status")["delivery_minutes"].agg(
    total_records="count",
    average_delivery="mean"
)

,total_records,average_delivery
status,,
cancelled,0,NaN
completed,10441,27.169716
pending,0,NaN


In [43]:
orders_clean.groupby("status")["delivery_minutes"].apply(
    lambda x: x.isna().sum()
)

status
cancelled    1199
completed     669
pending       656
Name: delivery_minutes, dtype: int64

Check completed orders with missing delivery time

In [44]:
completed_delivery_missing = orders_clean[
    (orders_clean["status"] == "completed") &
    (orders_clean["delivery_minutes"].isna())
]

print(completed_delivery_missing.shape)

completed_delivery_missing.head(10)

(669, 9)


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
22,ORD-108031,C0767,P018,3,2025-05-19 13:27:43,completed,20,144.00,NaN
23,ORD-103530,C0334,P005,3,2025-04-20 21:11:24,completed,15,122.40,NaN
31,ORD-104844,C0460,P001,2,2025-09-15 09:55:14,completed,20,206.40,NaN
118,ORD-111237,C1096,P010,1,2025-06-02 20:12:10,completed,5,42.75,NaN
124,ORD-100013,C0003,P024,2,2025-05-28 12:05:12,completed,15,153.00,NaN
146,ORD-111719,C1141,P016,2,2025-06-07 13:01:30,completed,0,50.00,NaN
174,ORD-109285,C0894,P005,1,2025-06-19 10:00:26,completed,25,36.00,NaN
247,ORD-111556,C1132,P026,2,2025-06-14 21:50:23,completed,0,380.00,NaN
282,ORD-103504,C0331,P038,1,2025-04-05 18:36:19,completed,40,1199.40,NaN
283,ORD-105442,C0530,P002,2,2025-06-12 22:04:57,completed,0,330.00,NaN


Check delivery time statistics

In [45]:
orders_clean["delivery_minutes"].describe()

count    10441.000000
mean        27.169716
std         13.662402
min        -10.000000
25%         19.000000
50%         25.000000
75%         31.000000
max         75.000000
Name: delivery_minutes, dtype: float64

Identify negative delivery times

In [46]:
negative_delivery = orders_clean[
    orders_clean["delivery_minutes"] < 0
]

print("Negative Delivery Records:", len(negative_delivery))

negative_delivery[
    ["order_id", "status", "delivery_minutes"]
]

Negative Delivery Records: 105


,order_id,status,delivery_minutes
180,ORD-111408,completed,-10.0
190,ORD-109808,completed,-10.0
348,ORD-101308,completed,-10.0
754,ORD-102908,completed,-10.0
771,ORD-109208,completed,-10.0
...,...,...,...
12684,ORD-107908,completed,-10.0
12798,ORD-104708,completed,-10.0
12802,ORD-109708,completed,-10.0
12932,ORD-100308,completed,-10.0


In [47]:
negative_delivery["status"].value_counts()

status
completed    105
Name: count, dtype: int64

Replace negative delivery times with NaN

In [48]:
orders_clean.loc[
    orders_clean["delivery_minutes"] < 0,
    "delivery_minutes"
] = np.nan

In [49]:
print("Negative Values Remaining:",
      (orders_clean["delivery_minutes"] < 0).sum())

print("Total Missing Delivery Times:",
      orders_clean["delivery_minutes"].isna().sum())

Negative Values Remaining: 0
Total Missing Delivery Times: 2629


Inspect completed orders with missing delivery times

In [50]:
completed_missing_delivery = orders_clean[
    (orders_clean["status"] == "completed") &
    (orders_clean["delivery_minutes"].isna())
]

print("Completed Orders with Missing Delivery Time:",
      len(completed_missing_delivery))

completed_missing_delivery.head()

Completed Orders with Missing Delivery Time: 774


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
22,ORD-108031,C0767,P018,3,2025-05-19 13:27:43,completed,20,144.00,NaN
23,ORD-103530,C0334,P005,3,2025-04-20 21:11:24,completed,15,122.40,NaN
31,ORD-104844,C0460,P001,2,2025-09-15 09:55:14,completed,20,206.40,NaN
118,ORD-111237,C1096,P010,1,2025-06-02 20:12:10,completed,5,42.75,NaN
124,ORD-100013,C0003,P024,2,2025-05-28 12:05:12,completed,15,153.00,NaN


Analyze Missing Delivery Times

In [51]:
completed_missing_delivery["order_ts"].describe()

count                           774
mean     2025-07-02 14:37:32.468992
min             2025-04-01 09:37:17
25%      2025-05-16 17:56:53.500000
50%             2025-07-02 18:43:06
75%      2025-08-18 09:45:53.750000
max             2025-09-30 21:27:46
Name: order_ts, dtype: object

In [52]:
completed_missing_delivery["quantity"].value_counts()

quantity
1    298
2    285
3    191
Name: count, dtype: int64

Check missing values by month

In [53]:
completed_missing_delivery["order_month"] = (completed_missing_delivery["order_ts"].dt.to_period('M'))

completed_missing_delivery["order_month"].value_counts().sort_index()

order_month
2025-04    114
2025-05    149
2025-06    120
2025-07    118
2025-08    146
2025-09    127
Freq: M, Name: count, dtype: int64

Verify final delivery-time quality

In [54]:
print("Negative values:",
      (orders_clean["delivery_minutes"] < 0).sum())

print("\nMissing delivery times by status:")
print(
    orders_clean.groupby("status")["delivery_minutes"]
    .apply(lambda x: x.isna().sum())
)

print("\nFinal delivery time statistics:")
print(orders_clean["delivery_minutes"].describe())

Negative values: 0

Missing delivery times by status:
status
cancelled    1199
completed     774
pending       656
Name: delivery_minutes, dtype: int64

Final delivery time statistics:
count    10336.000000
mean        27.547310
std         13.205249
min          8.000000
25%         19.000000
50%         25.000000
75%         32.000000
max         75.000000
Name: delivery_minutes, dtype: float64


In [55]:
orders_clean.info()

<class 'pandas.DataFrame'>
Index: 12965 entries, 0 to 13083
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          12965 non-null  str           
 1   customer_id       12965 non-null  str           
 2   product_id        12965 non-null  str           
 3   quantity          12965 non-null  int64         
 4   order_ts          12965 non-null  datetime64[us]
 5   status            12965 non-null  str           
 6   discount_pct      12965 non-null  int64         
 7   order_amount      12959 non-null  float64       
 8   delivery_minutes  10336 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(2), str(4)
memory usage: 1.5 MB


One final validation

In [56]:
print("Dataset Shape:", orders_clean.shape)

print("\nExact Duplicate Rows:",
      orders_clean.duplicated().sum())

print("\nDuplicate Order IDs:",
      orders_clean["order_id"].duplicated().sum())

print("\nMissing Values:")
print(orders_clean.isnull().sum())

Dataset Shape: (12965, 9)

Exact Duplicate Rows: 0

Duplicate Order IDs: 656

Missing Values:
order_id               0
customer_id            0
product_id             0
quantity               0
order_ts               0
status                 0
discount_pct           0
order_amount           6
delivery_minutes    2629
dtype: int64


In [57]:
orders_clean.to_csv("orders_cleaned.csv", index=False)

print("Orders dataset saved successfully!")

Orders dataset saved successfully!


## Orders Dataset – Data Cleaning Summary

Dataset Overview

- Original Records: 13,084
- Original Columns: 9
- Exact Duplicate Records: 119
- Records After Duplicate Removal: 12,965

Cleaning Operations Performed

1\. Duplicate Records: Identified and removed 119 exact duplicate rows. Preserved repeated Order IDs because they represent different order status history records.

2\. Order Amount: Converted the string column into numeric format. Identified 53 missing values and 65 invalid values (`"error"`). Recovered 112 completed order amounts using product reference prices and retained 6 missing pending order amounts.

3\. Order Timestamp: Converted the string column into datetime format. No invalid timestamps were identified.

4\. Order Status: Validated order status categories: Completed, Pending and Cancelled. No standardization changes were required.

5\. Quantity: Validated quantity values. All records contained valid quantities between 1 and 3.

6\. Discount Percentage: Validated discount values. All records contained values between 0% and 40%.

7\. Delivery Time: Identified 105 negative delivery time values and converted them into missing values. Analyzed missing delivery times across order statuses, months and quantities. Retained missing values without artificial imputation.

Final Validation

- Total Records: 12,965
- Total Columns: 9
- Exact Duplicate Records: 0
- Duplicate Order IDs: 656 (preserved as order history)
- Missing Order Amounts: 6
- Missing Delivery Times: 2,629
- Invalid Timestamps: 0
- Negative Delivery Time Values: 0
- Order Timestamp Data Type: Datetime
- Order Amount Data Type: Float
- Delivery Time Data Type: Float

Conclusion: The Orders dataset has undergone data cleaning, validation and missing-value analysis. Exact duplicate records and invalid delivery times were handled, while legitimate order history and unresolved missing values were preserved. The cleaned dataset is ready for subsequent data integration and analysis, subject to confirming the business rule used for order amount recovery.

Orders Dataset Cleaning

In [58]:
print("Shape:", orders.shape)

print("\nColumns:")
print(orders.columns.tolist())

print("\nMissing Values:")
print(orders.isnull().sum())

print("\nDuplicate Rows:", orders.duplicated().sum())

print("\nOrder Status Counts:")
print(orders["status"].value_counts())

Shape: (13084, 9)

Columns:
['order_id', 'customer_id', 'product_id', 'quantity', 'order_ts', 'status', 'discount_pct', 'order_amount', 'delivery_minutes']

Missing Values:
order_id               0
customer_id            0
product_id             0
quantity               0
order_ts               0
status                 0
discount_pct           0
order_amount          53
delivery_minutes    2541
dtype: int64

Duplicate Rows: 119

Order Status Counts:
status
completed    11217
cancelled     1211
pending        656
Name: count, dtype: int64


In [59]:
print("Unique Order IDs:", orders["order_id"].nunique())

print(
    "Duplicate Order IDs:",
    orders["order_id"].duplicated().sum()
)

print(
    "Unparseable Order Amounts:",
    pd.to_numeric(
        orders["order_amount"],
        errors="coerce"
    ).isna().sum()
)

print(
    "Negative Delivery Minutes:",
    (orders["delivery_minutes"] < 0).sum()
)

print(
    "Blank Delivery Minutes:",
    orders["delivery_minutes"].isna().sum()
)

Unique Order IDs: 12309
Duplicate Order IDs: 775
Unparseable Order Amounts: 120
Negative Delivery Minutes: 106
Blank Delivery Minutes: 2541


Convert DateTime

In [60]:
orders_clean = orders.copy()

orders_clean["order_ts"] = pd.to_datetime(
    orders_clean["order_ts"],
    errors="coerce"
)

print(orders_clean["order_ts"].dtype)
print("Missing dates:", orders_clean["order_ts"].isna().sum())

datetime64[us]
Missing dates: 0


Remove Exact Duplicate Rows

In [61]:
before = len(orders_clean)

orders_clean = orders_clean.drop_duplicates()

after = len(orders_clean)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)

Rows before: 13084
Rows after: 12965
Duplicates removed: 119


Keep Latest Record per Order ID

In [62]:
orders_clean = (
    orders_clean
    .sort_values("order_ts")
    .drop_duplicates(
        subset="order_id",
        keep="last"
    )
    .reset_index(drop=True)
)

print("Final Rows:", len(orders_clean))
print("Unique Order IDs:", orders_clean["order_id"].nunique())
print("Duplicate Order IDs:", orders_clean["order_id"].duplicated().sum())

Final Rows: 12309
Unique Order IDs: 12309
Duplicate Order IDs: 0


Amount =Price * Quantity (1- Discount / 100)

In [66]:
# Convert order_amount to numeric
orders_clean["order_amount"] = pd.to_numeric(
    orders_clean["order_amount"],
    errors="coerce"
)

# Get product prices using product_id
price_map = products_ref.set_index("product_id")["price_inr"]

# Calculate missing order amounts
calculated_amount = (
    orders_clean["product_id"].map(price_map)
    * orders_clean["quantity"]
    * (1 - orders_clean["discount_pct"] / 100)
).round(2)

# Fill missing amounts
orders_clean["order_amount"] = (
    orders_clean["order_amount"].fillna(calculated_amount)
)

print("Missing Order Amounts:", orders_clean["order_amount"].isna().sum())

Missing Order Amounts: 0


Clean Delivery Minutes

In [67]:
# Check status counts before cleaning delivery time
print(orders_clean["status"].value_counts())

# Convert negative delivery times to missing
orders_clean.loc[
    orders_clean["delivery_minutes"] < 0,
    "delivery_minutes"
] = pd.NA

print(
    "\nMissing Delivery Minutes:",
    orders_clean["delivery_minutes"].isna().sum()
)

status
completed    11110
cancelled     1199
Name: count, dtype: int64

Missing Delivery Minutes: 1973


In [68]:
print(
    orders_clean.groupby("status")["delivery_minutes"]
    .apply(lambda x: x.isna().sum())
)

status
cancelled    1199
completed     774
Name: delivery_minutes, dtype: int64


Category-wise Median Imputation

In [69]:
# Add product category to orders
orders_clean = orders_clean.merge(
    products_ref[["product_id", "category"]],
    on="product_id",
    how="left"
)

# Calculate category-wise median from completed orders
completed_median = (
    orders_clean.loc[
        orders_clean["status"] == "completed"
    ]
    .groupby("category")["delivery_minutes"]
    .transform("median")
)

# Fill missing delivery times only for completed orders
completed_mask = orders_clean["status"] == "completed"

orders_clean.loc[completed_mask, "delivery_minutes"] = (
    orders_clean.loc[completed_mask, "delivery_minutes"]
    .fillna(completed_median)
)

# Validate
print(
    orders_clean.groupby("status")["delivery_minutes"]
    .apply(lambda x: x.isna().sum())
)

status
cancelled    1199
completed       0
Name: delivery_minutes, dtype: int64


Final Orders Validation

In [70]:
print("Final Shape:", orders_clean.shape)

print("\nStatus Counts:")
print(orders_clean["status"].value_counts())

print("\nMissing Values:")
print(orders_clean.isnull().sum())

print("\nDuplicate Order IDs:",
      orders_clean["order_id"].duplicated().sum())

print("\nCompleted Revenue:",
      round(
          orders_clean.loc[
              orders_clean["status"] == "completed",
              "order_amount"
          ].sum(),
          2
      ))

Final Shape: (12309, 10)

Status Counts:
status
completed    11110
cancelled     1199
Name: count, dtype: int64

Missing Values:
order_id               0
customer_id            0
product_id             0
quantity               0
order_ts               0
status                 0
discount_pct           0
order_amount           0
delivery_minutes    1199
category               0
dtype: int64

Duplicate Order IDs: 0

Completed Revenue: 2998668.8


In [71]:
# Keep the original 9 order columns
order_columns = [
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "order_ts",
    "status",
    "discount_pct",
    "order_amount",
    "delivery_minutes"
]

orders_final = orders_clean[order_columns].copy()

# Save cleaned orders
orders_final.to_csv(
    "../data/clean/orders.csv",
    index=False
)

print("Orders dataset saved successfully!")
print("Final Shape:", orders_final.shape)

Orders dataset saved successfully!
Final Shape: (12309, 9)
